# Forecasting one InSAR ground-motion series with a small TCN

This notebook forecasts **one displacement time series** from its own history. The input is a pandas `DataFrame` with a `DatetimeIndex` and exactly one numeric column (ground motion in mm). It generates a reproducible example with more than 300 observations; replace one cell with your own `df` to use measured data.

The model is a causal, dilated 1D convolutional network (TCN) with a direct multi-step forecasting head. It predicts future displacements **relative to the last observed displacement**. The notebook chooses the input window, chronological partitions and batch size from the number of observations. The forecast horizon is a configurable number of acquisitions.

**Design limits:** No spatial neighbours or external covariates; observations are treated as approximately equally spaced in acquisition order. Future weather, missing values, uncertainty and spatial dependence need separate treatment. A prediction is not evidence of geophysical causation.


## 0. Requirements and reproducibility

Run in a Python environment with `numpy`, `pandas`, `matplotlib` and `torch`. In a notebook environment, install missing packages with `%pip install numpy pandas matplotlib torch`, then restart the kernel. For GPU-specific PyTorch builds, follow the official PyTorch installation selector for your system. CPU training is sufficient for the example.


In [ ]:
import copy
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__}; device: {DEVICE}")


## 1. Supply a DataFrame

The synthetic signal combines a velocity, an annual oscillation, a later change in velocity and measurement noise. Its construction is only for a runnable demonstration; results on it do not establish performance on real PS-InSAR data. Replace the next cell with your own `df`, for example `df = pd.read_csv("ground_motion.csv", parse_dates=["date"], index_col="date")`. Keep exactly one displacement column in millimetres.


In [ ]:
n_example = 540
dates = pd.date_range("2017-01-01", periods=n_example, freq="6D")
elapsed_days = (dates - dates[0]).days.to_numpy()
rng = np.random.default_rng(SEED)
displacement = (
    -2.2 * elapsed_days / 365.25
    + 3.0 * np.sin(2 * np.pi * elapsed_days / 365.25 + 0.3)
    - 1.0 * np.maximum(elapsed_days - 1650, 0) / 365.25
    + rng.normal(0, 0.65, n_example)
)
df = pd.DataFrame({"ground_motion_mm": displacement}, index=dates)
df.index.name = "date"
df.head()


## 2. Check input and choose split/window sizes

All boundaries are based on target dates. Training targets end before validation; validation targets end before test. At a validation or test forecast origin, earlier actual observations may be used as history, as they would be available at that time. No test target enters training or early stopping.

The default lookback is at most 120 acquisitions and at most about 55% of the training segment. This preserves enough training windows even near the 300-observation minimum; it is a practical starting rule, not an optimized hyperparameter. A five-block TCN with two kernel-3 convolutions per block has a theoretical receptive field of 125 observations.


In [ ]:
HORIZON = 6  # number of future acquisitions; change as needed
MIN_OBSERVATIONS = 300

def prepare_series(frame):
    if not isinstance(frame, pd.DataFrame) or frame.shape[1] != 1:
        raise ValueError("Provide a pandas DataFrame with exactly one displacement column.")
    if not isinstance(frame.index, pd.DatetimeIndex):
        raise TypeError("The DataFrame index must be a DatetimeIndex.")
    if frame.index.hasnans or frame.index.has_duplicates:
        raise ValueError("Dates must be valid and unique.")
    series = pd.to_numeric(frame.iloc[:, 0], errors="raise").sort_index().astype("float64")
    if len(series) < MIN_OBSERVATIONS:
        raise ValueError(f"Need at least {MIN_OBSERVATIONS} observations; got {len(series)}.")
    if not np.isfinite(series.to_numpy()).all():
        raise ValueError("The displacement column must have no missing or infinite values. Handle gaps explicitly first.")
    gaps = series.index.to_series().diff().dropna()
    median_gap = gaps.median()
    if median_gap <= pd.Timedelta(0):
        raise ValueError("Dates must be strictly increasing after sorting.")
    if gaps.max() > 1.5 * median_gap:
        warnings.warn("Acquisition gaps are irregular; the CNN models acquisition steps, not elapsed days. Review the time axis before interpreting forecasts.")
    return series, median_gap

series, median_gap = prepare_series(df)
values = series.to_numpy()
n = len(values)
train_end = int(0.65 * n)
val_end = int(0.80 * n)
lookback = min(120, max(36, int(0.55 * train_end)))
if not isinstance(HORIZON, int) or HORIZON < 1:
    raise ValueError("HORIZON must be a positive integer.")
if train_end - lookback - HORIZON + 1 < 20 or val_end - train_end < HORIZON or n - val_end < HORIZON:
    raise ValueError("Not enough windows for this forecast horizon. Shorten HORIZON or provide more observations.")

# Train-only scale; each window is centred on its own final observed value.
scale_mm = max(float(np.std(np.diff(values[:train_end])) * np.sqrt(lookback)), 1.0)
print(f"N={n}, train targets < {train_end}, validation targets {train_end}:{val_end}, test targets >= {val_end}")
print(f"Lookback={lookback}, horizon={HORIZON}, typical cadence={median_gap}, train-only scale={scale_mm:.2f} mm")


## 3. Build rolling-origin examples

A forecast origin `p` uses `values[p-lookback:p]` and predicts `values[p:p+horizon]`. A multi-step forecast is emitted at once. We do not precompute trend or seasonal components using observations beyond an origin.


In [ ]:
def make_windows(values, origins, lookback, horizon, scale_mm):
    xs, ys = [], []
    for p in origins:
        last = values[p - 1]
        xs.append(((values[p - lookback:p] - last) / scale_mm)[None, :])
        ys.append((values[p:p + horizon] - last) / scale_mm)
    return np.asarray(xs, dtype=np.float32), np.asarray(ys, dtype=np.float32)

train_origins = np.arange(lookback, train_end - HORIZON + 1)
val_origins = np.arange(train_end, val_end - HORIZON + 1)
test_origins = np.arange(val_end, n - HORIZON + 1)
X_train, y_train = make_windows(values, train_origins, lookback, HORIZON, scale_mm)
X_val, y_val = make_windows(values, val_origins, lookback, HORIZON, scale_mm)
X_test, y_test = make_windows(values, test_origins, lookback, HORIZON, scale_mm)

batch_size = min(64, max(8, len(X_train) // 8))
train_loader = DataLoader(TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train)), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val)), batch_size=batch_size, shuffle=False)
print(f"Windows: train={len(X_train)}, validation={len(X_val)}, test={len(X_test)}; batch size={batch_size}")
assert train_origins[-1] + HORIZON <= train_end
assert val_origins[0] >= train_end and val_origins[-1] + HORIZON <= val_end
assert test_origins[0] >= val_end and test_origins[-1] + HORIZON <= n


## 4. Define a compact causal TCN

Causality keeps hidden features at time `t` dependent only on inputs through `t`. All input samples already end at the forecast origin; the direct head maps the final hidden state to all future steps. The residual blocks use dilations 1, 2, 4, 8 and 16.


In [ ]:
class CausalConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation):
        super().__init__()
        self.pad = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size, padding=self.pad, dilation=dilation)

    def forward(self, x):
        y = self.conv(x)
        return y[..., :-self.pad] if self.pad else y

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dilation, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            CausalConv1d(in_channels, out_channels, 3, dilation), nn.ReLU(), nn.Dropout(dropout),
            CausalConv1d(out_channels, out_channels, 3, dilation), nn.ReLU(), nn.Dropout(dropout),
        )
        self.skip = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        return torch.relu(self.net(x) + self.skip(x))

class GroundMotionTCN(nn.Module):
    def __init__(self, horizon, channels=32):
        super().__init__()
        blocks = []
        in_channels = 1
        for dilation in (1, 2, 4, 8, 16):
            blocks.append(ResidualBlock(in_channels, channels, dilation))
            in_channels = channels
        self.features = nn.Sequential(*blocks)
        self.head = nn.Linear(channels, horizon)

    def forward(self, x):
        return self.head(self.features(x)[:, :, -1])

model = GroundMotionTCN(HORIZON).to(DEVICE)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


## 5. Train with early stopping

Validation targets guide epoch selection. The test segment stays untouched until the following evaluation. The seed reduces run-to-run variation but exact GPU reproducibility is not guaranteed.


In [ ]:
def mean_loss(model, loader, loss_fn):
    model.eval()
    total, count = 0.0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            total += loss_fn(model(xb), yb).item() * len(xb)
            count += len(xb)
    return total / count

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.MSELoss()
max_epochs, patience = 80, 12
best_val = float("inf")
best_state = None
wait = 0
history = []

for epoch in range(1, max_epochs + 1):
    model.train()
    total = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item() * len(xb)
    train_mse = total / len(X_train)
    val_mse = mean_loss(model, val_loader, loss_fn)
    history.append((epoch, train_mse, val_mse))
    if val_mse < best_val - 1e-6:
        best_val, best_state, wait = val_mse, copy.deepcopy(model.state_dict()), 0
    else:
        wait += 1
    if epoch == 1 or epoch % 10 == 0 or wait == patience:
        print(f"Epoch {epoch:3d} | train MSE {train_mse:.4f} | val MSE {val_mse:.4f}")
    if wait >= patience:
        print("Early stopping")
        break

model.load_state_dict(best_state)
model.eval()
history_df = pd.DataFrame(history, columns=["epoch", "train_mse", "val_mse"]).set_index("epoch")
history_df.plot(title="Training history (scaled displacement MSE)", figsize=(8, 3))
plt.tight_layout()
plt.show()


## 6. Evaluate against persistence and local linear extrapolation

For each test origin, persistence repeats the last observed displacement. The linear baseline fits the latest 30 **available** observations to actual elapsed days and extrapolates to the target dates. Both baselines, the CNN and the metrics use identical test origins and targets.

Overlapping test windows share observations; the pooled scores describe these forecast decisions and are **not** independent-sample confidence intervals. For model selection across sites, use additional geographic and temporal holdouts.


In [ ]:
@torch.no_grad()
def predict_tcn(model, X, origins, values, scale_mm):
    model.eval()
    chunks = []
    for start in range(0, len(X), 256):
        xb = torch.from_numpy(X[start:start + 256]).to(DEVICE)
        chunks.append(model(xb).cpu().numpy())
    relative = np.concatenate(chunks) * scale_mm
    return values[origins - 1, None] + relative

def predict_baselines(series, origins, horizon, linear_lookback=30):
    values = series.to_numpy()
    dates = series.index
    persistence = np.repeat(values[origins - 1, None], horizon, axis=1)
    linear = np.empty_like(persistence)
    for row, p in enumerate(origins):
        past_dates = dates[p - linear_lookback:p]
        x = (past_dates - past_dates[0]).total_seconds().to_numpy() / 86400.0
        future_x = (dates[p:p + horizon] - past_dates[0]).total_seconds().to_numpy() / 86400.0
        slope, intercept = np.polyfit(x, values[p - linear_lookback:p], 1)
        linear[row] = intercept + slope * future_x
    return persistence, linear

truth = values[test_origins[:, None] + np.arange(HORIZON)]
predictions = {"TCN": predict_tcn(model, X_test, test_origins, values, scale_mm)}
predictions["Persistence"], predictions["Linear (30 obs.)"] = predict_baselines(series, test_origins, HORIZON)

rows = []
for name, pred in predictions.items():
    error = pred - truth
    rows.append({"model": name, "MAE_mm": np.abs(error).mean(), "RMSE_mm": np.sqrt(np.mean(error**2)), "bias_mm": error.mean()})
metrics = pd.DataFrame(rows).set_index("model").sort_values("MAE_mm")
print("Pooled test errors across all origins and horizons (mm)")
display(metrics.round(3))

horizon_mae = pd.DataFrame(
    {name: np.abs(pred - truth).mean(axis=0) for name, pred in predictions.items()},
    index=np.arange(1, HORIZON + 1),
)
horizon_mae.index.name = "lead_acquisitions"
display(horizon_mae.round(3))
horizon_mae.plot(marker="o", ylabel="MAE (mm)", title="Test MAE by forecast lead", figsize=(8, 3))
plt.tight_layout()
plt.show()


## 7. Inspect one held-out forecast

The selected origin is the final evaluable origin in the test segment, so its entire target path is known for comparison. This plot is a diagnostic example, not the aggregate test result.


In [ ]:
row = -1
p = int(test_origins[row])
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(series.index[p - lookback:p], values[p - lookback:p], label="Observed history", color="0.35")
ax.plot(series.index[p:p + HORIZON], truth[row], "o-", label="Actual future", color="black")
for name, pred in predictions.items():
    ax.plot(series.index[p:p + HORIZON], pred[row], "o--", label=name)
ax.axvline(series.index[p - 1], color="0.5", linestyle=":")
ax.set(xlabel="Date", ylabel="LOS ground motion (mm)", title=f"Test forecast from {series.index[p - 1].date()}")
ax.legend(ncol=2)
plt.tight_layout()
plt.show()


## 8. Forecast beyond the last observed acquisition

This uses the selected model trained only on the training segment (with validation used for early stopping) and the latest observed input window. For the illustrative future dates, it repeats the median acquisition interval. Those dates are **estimates**, not a Sentinel-1 schedule. No actual future displacement is available for scoring. For a real deployment, define an acquisition calendar and retraining policy explicitly.


In [ ]:
last = values[-1]
latest_x = ((values[-lookback:] - last) / scale_mm)[None, None, :].astype(np.float32)
with torch.no_grad():
    future_relative = model(torch.from_numpy(latest_x).to(DEVICE)).cpu().numpy()[0] * scale_mm
future_dates = pd.DatetimeIndex([series.index[-1] + k * median_gap for k in range(1, HORIZON + 1)], name=series.index.name)
forecast = pd.DataFrame({"forecast_ground_motion_mm": last + future_relative}, index=future_dates)
display(forecast.round(3))

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(series.iloc[-2 * lookback:].index, series.iloc[-2 * lookback:].values, label="Observed")
ax.plot(pd.DatetimeIndex([series.index[-1], *future_dates]), np.r_[last, forecast.iloc[:, 0].to_numpy()], "o--", label="TCN forecast")
ax.set(xlabel="Date", ylabel="LOS ground motion (mm)")
ax.legend()
plt.tight_layout()
plt.show()


## When replacing the example with PS-InSAR data

- Use one consistently referenced LOS displacement series in mm, with a unique date for each acquisition and at least 300 finite measurements. Do not silently fill missing data using future measurements.
- Review acquisition gaps, reference changes, atmospheric artifacts and jumps. This notebook predicts in acquisition steps, so a six-step horizon may represent different calendar durations when sampling changes.
- Compare the **test** metrics to persistence and linear extrapolation; if the CNN does not beat them, it has not shown useful predictive skill on this series.
- A single long PS series yields strongly dependent rolling windows. Results from one series cannot establish generalization across scatterers or sites. A multivariate model with groundwater or weather histories should be evaluated only after the single-series protocol works reliably.
